# SONA MS-SNSD Strict Guard Pilot

Second guarded pilot after the first output-drift guard reduced but did not remove PESQ/LSD degradation. This tests no-update rows and stricter output guards on 300 MS-SNSD clips.


In [ ]:
%pip install -q pystoi pesq ptflops

import glob
import os
import sys
from pathlib import Path

IN_KAGGLE = Path('/kaggle/working').exists()
print('IN_KAGGLE =', IN_KAGGLE)

In [ ]:
def first_existing(candidates):
    for c in candidates:
        if c and Path(c).exists():
            return str(Path(c))
    return None

def find_repo():
    here = Path.cwd()
    if (here / 'scripts' / 'eval_external_ms_snsd.py').exists():
        return str(here)
    hits = glob.glob('/kaggle/input/**/scripts/eval_external_ms_snsd.py', recursive=True)
    return str(Path(sorted(hits, key=len)[0]).parents[1]) if hits else None

def find_manifest():
    local = r'D:\noise cancellation\external_data\ms_snsd_sona\test.csv'
    if Path(local).exists():
        return local
    hits = [p for p in glob.glob('/kaggle/input/**/test.csv', recursive=True) if 'ms' in p.lower() and 'snsd' in p.lower()]
    return sorted(hits, key=len)[0] if hits else None

def find_checkpoint():
    local = r'D:\noise cancellation\results\kaggle_online_tune_2026-07-04_full824_trust_region\extracted\ckpt_attention\last.pt'
    if Path(local).exists():
        return local
    hits = glob.glob('/kaggle/input/**/ckpt_attention/last.pt', recursive=True)
    return sorted(hits, key=len)[0] if hits else None

REPO_DIR = find_repo()
MANIFEST = find_manifest()
CHECKPOINT = find_checkpoint()
OUT_DIR = '/kaggle/working/ms_snsd_external_eval' if IN_KAGGLE else r'D:\noise cancellation\results\ms_snsd_external_eval'

# If auto-detect fails, set these manually:
# REPO_DIR = r'D:\noise cancellation'
# MANIFEST = r'D:\noise cancellation\external_data\ms_snsd_sona\test.csv'
# CHECKPOINT = r'D:\noise cancellation\results\kaggle_online_tune_2026-07-04_full824_trust_region\extracted\ckpt_attention\last.pt'

assert REPO_DIR, 'Repo with scripts/eval_external_ms_snsd.py not found.'
assert MANIFEST, 'MS-SNSD manifest test.csv not found.'
assert CHECKPOINT, 'ckpt_attention/last.pt not found.'

print('REPO_DIR   =', REPO_DIR)
print('MANIFEST   =', MANIFEST)
print('CHECKPOINT =', CHECKPOINT)
print('OUT_DIR    =', OUT_DIR)

In [ ]:
# Strict guarded MS-SNSD pilot. Keep LIMIT=300 first.
# STEPS includes 0, which is an explicit no-update row for sanity.
LIMIT = 300
ADAPTER = 'lowrank'
ANCHORS = '2.0'
STEPS = '0,1'
MAX_GAIN_DELTAS = '0.01,0.02'
GUARD_MIN_REWARD_DELTAS = 'none'
GUARD_MAX_OUTPUT_DELTAS = '0.0,0.0001,0.00025,0.0005'
REWARD_KIND = 'auto'

print(dict(
    LIMIT=LIMIT,
    ADAPTER=ADAPTER,
    ANCHORS=ANCHORS,
    STEPS=STEPS,
    MAX_GAIN_DELTAS=MAX_GAIN_DELTAS,
    GUARD_MIN_REWARD_DELTAS=GUARD_MIN_REWARD_DELTAS,
    GUARD_MAX_OUTPUT_DELTAS=GUARD_MAX_OUTPUT_DELTAS,
    REWARD_KIND=REWARD_KIND,
))


In [ ]:
import subprocess

cmd = [
    sys.executable, str(Path(REPO_DIR) / 'scripts' / 'eval_external_ms_snsd.py'),
    '--repo', REPO_DIR,
    '--manifest', MANIFEST,
    '--checkpoint', CHECKPOINT,
    '--out', OUT_DIR,
    '--limit', str(LIMIT),
    '--adapter', ADAPTER,
    '--anchors', ANCHORS,
    '--steps', STEPS,
    '--max-gain-deltas', MAX_GAIN_DELTAS,
    '--guard-min-reward-deltas', GUARD_MIN_REWARD_DELTAS,
    '--guard-max-output-deltas', GUARD_MAX_OUTPUT_DELTAS,
    '--reward-kind', REWARD_KIND,
]

print(' '.join(cmd))
print()
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
assert proc.returncode == 0, f'eval_external_ms_snsd.py exited {proc.returncode}'


In [ ]:
import pandas as pd

summary_md = Path(OUT_DIR) / 'summary.md'
summary_csv = Path(OUT_DIR) / 'summary.csv'
print(summary_md.read_text(encoding='utf-8'))
df = pd.read_csv(summary_csv)
display(df.sort_values(['pesq', 'si_sdr_i'], ascending=[False, False]))

base = df[df.method == 'neural_attention'].iloc[0]
online = df[df.method.str.startswith('attention_online_')].copy()
for col in ['si_sdr_i', 'pesq', 'stoi', 'estoi', 'lsd']:
    if col in online and col in base:
        online['delta_' + col] = online[col] - base[col]
display(online.sort_values(['delta_pesq', 'delta_si_sdr_i'], ascending=[False, False]))

In [ ]:
import shutil
from IPython.display import FileLink, display

ART = '/kaggle/working/sona_ms_snsd_external_eval' if IN_KAGGLE else r'D:\noise cancellation\results\sona_ms_snsd_external_eval'
shutil.make_archive(ART, 'zip', OUT_DIR)
print(f'Wrote {ART}.zip ({os.path.getsize(ART + ".zip") / 1e6:.1f} MB)')
display(FileLink(ART + '.zip'))